In [1]:
import pandas as pd
import numpy as np
import random
import time

# Read input data
data = pd.read_excel('data/input42u.xlsx')
a = data['a'].values
b = data['b'].values
c = data['c'].values
Minimum_Capacity = data['pmin'].values
Maximum_Capacity = data['pmax'].values
ramp_up = data['rampup'].values
ramp_down = data['rampdown'].values
Unit = data['Unit'].values

# Read power demand data
power_demand_data = pd.read_excel('data/power_demand42u.xlsx')
power_demand = power_demand_data['load'].values

# ABC Parameters
num_bees = 50
num_iterations = 100
limit = 10  # Limit for abandonment

# Define ABC-based power calculation
def calculate_power(demand, a, b, c, pmin, pmax, rampup, rampdown):
    num_units = len(pmin)

    # Initialize bee positions (random solutions)
    bees = np.random.uniform(pmin, pmax, (num_bees, num_units))
    fitness = np.full(num_bees, float('inf'))
    trial_counter = np.zeros(num_bees)

    # Best solution
    best_bee = None
    best_score = float('inf')

    def evaluate_fitness(bee):
        squared_P = np.multiply(bee, bee)
        fuel_cost = np.sum(np.add(a, np.add(np.multiply(b, bee), np.multiply(c, squared_P))))
        power_output = np.sum(bee)
        mismatch_penalty = abs(demand - power_output) * 10000  # Large penalty for mismatch
        return fuel_cost + mismatch_penalty

    # Evaluate initial fitness
    for i in range(num_bees):
        bees[i] = np.clip(bees[i], pmin, pmax)
        fitness[i] = evaluate_fitness(bees[i])
        if fitness[i] < best_score:
            best_score = fitness[i]
            best_bee = bees[i].copy()

    for iteration in range(num_iterations):
        # Employed bees phase
        for i in range(num_bees):
            k = random.choice([x for x in range(num_bees) if x != i])
            phi = np.random.uniform(-1, 1, size=num_units)
            new_bee = bees[i] + phi * (bees[i] - bees[k])

            # Respect ramp rate constraints
            diff = new_bee - bees[i]
            ramp_up_mask = diff > 0
            ramp_down_mask = diff < 0
            ramp_up_values = np.minimum(diff[ramp_up_mask], rampup[ramp_up_mask])
            ramp_down_values = np.maximum(diff[ramp_down_mask], -rampdown[ramp_down_mask])
            new_bee[ramp_up_mask] = bees[i][ramp_up_mask] + ramp_up_values
            new_bee[ramp_down_mask] = bees[i][ramp_down_mask] + ramp_down_values

            # Ensure generation limits
            new_bee = np.clip(new_bee, pmin, pmax)

            new_fitness = evaluate_fitness(new_bee)
            if new_fitness < fitness[i]:
                bees[i] = new_bee
                fitness[i] = new_fitness
                trial_counter[i] = 0
            else:
                trial_counter[i] += 1

            # Update best solution
            if fitness[i] < best_score:
                best_score = fitness[i]
                best_bee = bees[i].copy()

        # Onlooker bees phase
        total_fitness = np.sum(1 / (1 + fitness))  # Convert to probabilities
        probs = (1 / (1 + fitness)) / total_fitness
        for _ in range(num_bees):
            i = np.random.choice(range(num_bees), p=probs)
            k = random.choice([x for x in range(num_bees) if x != i])
            phi = np.random.uniform(-1, 1, size=num_units)
            new_bee = bees[i] + phi * (bees[i] - bees[k])

            # Respect ramp rate constraints
            diff = new_bee - bees[i]
            ramp_up_mask = diff > 0
            ramp_down_mask = diff < 0
            ramp_up_values = np.minimum(diff[ramp_up_mask], rampup[ramp_up_mask])
            ramp_down_values = np.maximum(diff[ramp_down_mask], -rampdown[ramp_down_mask])
            new_bee[ramp_up_mask] = bees[i][ramp_up_mask] + ramp_up_values
            new_bee[ramp_down_mask] = bees[i][ramp_down_mask] + ramp_down_values

            # Ensure generation limits
            new_bee = np.clip(new_bee, pmin, pmax)

            new_fitness = evaluate_fitness(new_bee)
            if new_fitness < fitness[i]:
                bees[i] = new_bee
                fitness[i] = new_fitness
                trial_counter[i] = 0
            else:
                trial_counter[i] += 1

            # Update best solution
            if fitness[i] < best_score:
                best_score = fitness[i]
                best_bee = bees[i].copy()

        # Scout bees phase
        for i in range(num_bees):
            if trial_counter[i] >= limit:
                bees[i] = np.random.uniform(pmin, pmax, num_units)
                bees[i] = np.clip(bees[i], pmin, pmax)
                fitness[i] = evaluate_fitness(bees[i])
                trial_counter[i] = 0

                # Update best solution
                if fitness[i] < best_score:
                    best_score = fitness[i]
                    best_bee = bees[i].copy()

        # Normalize to meet demand
        total_power = np.sum(best_bee)
        if total_power != 0:
            scaling_factor = demand / total_power
            best_bee = best_bee * scaling_factor

        # Ensure best bee respects generation limits
        best_bee = np.clip(best_bee, pmin, pmax)

        # Termination condition
        if abs(demand - np.sum(best_bee)) < 0.0001:
            break

    return best_bee, iteration + 1

# Fuel cost calculation
def calculate_fuel_cost(P, a, b, c):
    squaredP = np.multiply(P, P)
    Fuel_Cost = np.divide(np.add(a, np.add(np.multiply(b, P), np.multiply(c, squaredP))), 1000)
    total_Fuel_Cost = np.sum(Fuel_Cost)
    return Fuel_Cost, total_Fuel_Cost

# Loop through power demand data and calculate results
all_outputs = []
schedulling = []

for load_counter, demand in enumerate(power_demand, start=1):
    start_time = time.time()
    if load_counter == 1:
        previous_output = None
    else:
        previous_output = all_outputs[-1]['Output']['Power Produced (MW)'].values
    P, iterations = calculate_power(demand, a, b, c, Minimum_Capacity, Maximum_Capacity, ramp_up, ramp_down)
    Fuel_Cost, total_Fuel_Cost = calculate_fuel_cost(P, a, b, c)
    end_time = time.time()

    # Output Results
    output = pd.DataFrame({'Unit': Unit, 'Power Produced (MW)': P})
    print("Load Counter: {}".format(load_counter))
    print(f"Output for Power Demand {demand:.2f} MW")
    print(output)
    print(f"Total Power Produced: {np.sum(P):.2f} MW")
    print(f'Total Fuel Cost: Rp {total_Fuel_Cost}')
    print(f'Computation Time: {end_time - start_time} seconds')
    print('------------------------------------------------\n')

    all_outputs.append({'Demand': demand, 
                        'Output': output, 
                        'Total Power Produced': np.sum(P),
                        'Total Fuel Cost': total_Fuel_Cost,  # No need to sum again, it's already a single value
                        'Computation Time': end_time - start_time})

    # Scheduling
    output_info = {'Demand': power_demand[load_counter - 1]}
    for unit_index, unit in enumerate(Unit):
        output_info[f'Unit {unit} Power Produced'] = P[unit_index]
        output_info[f'Unit {unit} Cost'] = total_Fuel_Cost  # Assign total cost directly
    schedulling.append(output_info)

# Calculate ramp rates and save to an additional sheet
ramp_rates = []
generation_limits = []
for i in range(1, len(power_demand)):  # Start from 1 because there's no previous output for the first demand
    previous_output = all_outputs[i-1]['Output']['Power Produced (MW)'].values
    current_output = all_outputs[i]['Output']['Power Produced (MW)'].values
    ramp_rate = current_output - previous_output

    # Check for ramp rate violations
    violations = np.logical_or(ramp_rate > ramp_up, ramp_rate < -ramp_down)
    ramp_rate_info = {'Demand': power_demand[i]}
    generation_limit_info = {'Demand': power_demand[i]}
    for unit_index, unit in enumerate(Unit):
        ramp_rate_info[f'Unit {unit}'] = ramp_rate[unit_index]
        ramp_rate_info[f'Unit {unit} Violation'] = violations[unit_index]
               
        # Check for generation limit violations
        gen_violation = (current_output[unit_index] < Minimum_Capacity[unit_index]) or (current_output[unit_index] > Maximum_Capacity[unit_index])
        generation_limit_info[f'Unit {unit}'] = current_output[unit_index]
        generation_limit_info[f'Unit {unit} Violation'] = gen_violation
    ramp_rates.append(ramp_rate_info)
    generation_limits.append(generation_limit_info)
# Save results to Excel
output_file_path = 'ABC (ED).xlsx'
with pd.ExcelWriter(output_file_path) as writer:
    pd.DataFrame(schedulling).to_excel(writer, sheet_name='Schedulling', index=False)
    pd.DataFrame(all_outputs).to_excel(writer, sheet_name='Summary', index=False)
    pd.DataFrame(ramp_rates).to_excel(writer, sheet_name='Ramp Rates', index=False)
    pd.DataFrame(generation_limits).to_excel(writer, sheet_name='Generation Limits', index=False)
print("All results saved to 'ABC (ED).xlsx'")

Load Counter: 1
Output for Power Demand 8761.42 MW
    Unit  Power Produced (MW)
0      1           264.271036
1      2           264.271036
2      3           271.557159
3      4           264.271036
4      5           431.290331
5      6           440.255687
6      7           431.290331
7      8            92.494863
8      9           169.551606
9     10           138.006539
10    11           174.600000
11    12            42.283366
12    13            80.000000
13    14            79.281311
14    15            79.522454
15    16           232.558512
16    17           235.956524
17    18           232.558512
18    19             8.734481
19    20             8.952544
20    21           108.639560
21    22            84.247917
22    23            86.203591
23    24           111.011607
24    25           105.246870
25    26           108.038890
26    27            63.425049
27    28            92.285689
28    29           432.822682
29    30           436.540300
30    31           